# NeuroScribe — EDA & Preprocessing
**Dataset:** CHB-MIT Scalp EEG Database  
**Goal:** Explore the dataset, visualize key statistics, and run the full preprocessing pipeline.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# ── Make src/ importable from notebooks/ ───────────────────────────────────
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.signal import welch, spectrogram
import mne
mne.set_log_level('WARNING')

# ── NeuroScribe src imports ─────────────────────────────────────────────────
from src.data.loader import (
    parse_summary_file,
    build_patient_manifest,
    load_recording,
    dataset_stats,
)
from src.data.preprocessor import (
    bandpass_filter,
    notch_filter,
    apply_filters,
    create_windows,
    zscore_normalize,
)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR    = '../data/chb-mit'
FS          = 256
WINDOW_SEC  = 4
OVERLAP     = 0.5
WINDOW_SAMP = WINDOW_SEC * FS
STEP_SAMP   = int(WINDOW_SAMP * (1 - OVERLAP))

print('Setup complete.')

---
## 1. Dataset Overview

In [ ]:
# ── Parse a summary.txt file ───────────────────────────────────────────────
def parse_summary(summary_path):
    """Returns {edf_filename: [(onset_sec, offset_sec), ...]}"""
    result, current, starts, ends = {}, None, {}, {}
    with open(summary_path) as f:
        for line in f:
            line = line.strip()
            m = re.match(r'File Name:\s+(\S+\.edf)', line, re.I)
            if m:
                if current:
                    result[current] = [(starts[i], ends[i]) for i in sorted(starts) if i in ends]
                current = m.group(1); starts, ends = {}, {}; result.setdefault(current, [])
                continue
            m = re.match(r'Seizure\s*(\d*)\s*Start Time:\s*(\d+)', line, re.I)
            if m: starts[int(m.group(1) or 1)] = float(m.group(2)); continue
            m = re.match(r'Seizure\s*(\d*)\s*End Time:\s*(\d+)', line, re.I)
            if m: ends[int(m.group(1) or 1)]   = float(m.group(2))
    if current:
        result[current] = [(starts[i], ends[i]) for i in sorted(starts) if i in ends]
    return result


# ── Scan all patient directories ───────────────────────────────────────────
rows = []
patient_dirs = sorted([d for d in os.listdir(DATA_DIR) if d.startswith('chb')])

for pdir in patient_dirs:
    ppath = os.path.join(DATA_DIR, pdir)
    summaries = [f for f in os.listdir(ppath) if 'summary' in f.lower() and f.endswith('.txt')]
    edfs      = [f for f in os.listdir(ppath) if f.endswith('.edf')]
    if not summaries: continue

    ann  = parse_summary(os.path.join(ppath, summaries[0]))
    sz_files  = [f for f, s in ann.items() if s]
    all_seiz  = [s for seiz in ann.values() for s in seiz]
    total_dur = sum(e - o for o, e in all_seiz)

    rows.append({
        'Patient':         pdir,
        'EDF Files':       len(edfs),
        'Seizure Files':   len(sz_files),
        'Total Seizures':  len(all_seiz),
        'Seizure Duration (s)': round(total_dur, 1),
        'Avg Duration (s)':     round(total_dur / max(len(all_seiz), 1), 1),
    })

df_overview = pd.DataFrame(rows)
print(f'Patients found : {len(df_overview)}')
print(f'Total EDF files: {df_overview["EDF Files"].sum()}')
print(f'Total seizures : {df_overview["Total Seizures"].sum()}')
print(f'Total seizure duration: {df_overview["Seizure Duration (s)"].sum():.0f} s')
print()
df_overview

In [ ]:
# ── Per-patient seizure count bar chart ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(df_overview['Patient'], df_overview['Total Seizures'], color='steelblue')
axes[0].set_xlabel('Patient'); axes[0].set_ylabel('Number of Seizures')
axes[0].set_title('Seizure Count per Patient')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(df_overview['Patient'], df_overview['Seizure Duration (s)'], color='tomato')
axes[1].set_xlabel('Patient'); axes[1].set_ylabel('Total Seizure Duration (s)')
axes[1].set_title('Total Seizure Duration per Patient')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 2. Load a Sample Recording

In [ ]:
# ── Pick first patient that has a seizure EDF on disk ─────────────────────
PATIENT    = 'chb01'
PATIENT_DIR = os.path.join(DATA_DIR, PATIENT)

ann = parse_summary(os.path.join(PATIENT_DIR, f'{PATIENT}-summary.txt'))
seizure_edfs = [f for f, s in ann.items() if s and os.path.exists(os.path.join(PATIENT_DIR, f))]
normal_edfs  = [f for f, s in ann.items() if not s and os.path.exists(os.path.join(PATIENT_DIR, f))]

print(f'Patient       : {PATIENT}')
print(f'Seizure EDFs on disk : {len(seizure_edfs)}')
print(f'Normal  EDFs on disk : {len(normal_edfs)}')
print(f'\nSeizure files:')
for f in seizure_edfs:
    ivs = ann[f]
    print(f'  {f}  →  ' + ', '.join(f'{o:.0f}–{e:.0f}s (dur {e-o:.0f}s)' for o, e in ivs))

In [ ]:
# ── Load first seizure EDF ─────────────────────────────────────────────────
TARGET_EDF = seizure_edfs[0]
edf_path   = os.path.join(PATIENT_DIR, TARGET_EDF)

raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
data_v, times = raw.get_data(return_times=True)
data = data_v * 1e6   # Volts → µV

n_channels, n_samples = data.shape
duration_sec = n_samples / FS

print(f'File          : {TARGET_EDF}')
print(f'Shape         : {data.shape}  (channels × samples)')
print(f'Sample rate   : {int(raw.info["sfreq"])} Hz')
print(f'Duration      : {duration_sec:.1f} s  ({duration_sec/60:.1f} min)')
print(f'Channels      : {n_channels}')
print(f'Value range   : [{data.min():.1f}, {data.max():.1f}] µV')
print(f'\nChannel names:')
for i, ch in enumerate(raw.ch_names):
    print(f'  [{i:02d}] {ch}')

In [ ]:
# ── Print first 5 raw samples ──────────────────────────────────────────────
n_show = min(6, n_channels)
df_samples = pd.DataFrame(
    data[:n_show, :5].T,
    columns=raw.ch_names[:n_show],
    index=[f't={i/FS:.4f}s' for i in range(5)]
).round(2)
print('First 5 samples (µV) — first 6 channels:')
df_samples

---
## 3. EDA — Signal Visualization

In [ ]:
# ── Build sample-level seizure label array ─────────────────────────────────
seizure_intervals = ann[TARGET_EDF]
label_array = np.zeros(n_samples, dtype=np.int8)
for onset, offset in seizure_intervals:
    label_array[int(onset*FS) : int(offset*FS)] = 1

print(f'Seizure intervals : {seizure_intervals}')
print(f'Seizure samples   : {label_array.sum()} / {n_samples}  ({100*label_array.mean():.2f}%)')

In [ ]:
# ── Full recording overview with seizure shading ───────────────────────────
CH_SHOW = min(6, n_channels)
offset_scale = 300   # µV offset between channels for visibility

fig, ax = plt.subplots(figsize=(16, 7))
t = np.arange(n_samples) / FS

for i in range(CH_SHOW):
    ax.plot(t, data[i] + i * offset_scale, lw=0.4, color='steelblue', alpha=0.8)

for onset, offset in seizure_intervals:
    ax.axvspan(onset, offset, alpha=0.25, color='red', label='Seizure')

ax.set_yticks([i * offset_scale for i in range(CH_SHOW)])
ax.set_yticklabels(raw.ch_names[:CH_SHOW], fontsize=8)
ax.set_xlabel('Time (s)'); ax.set_title(f'{PATIENT} — {TARGET_EDF} — Full Recording (first {CH_SHOW} channels)')
handles, labels_ = ax.get_legend_handles_labels()
if handles: ax.legend(handles[:1], labels_[:1], loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# ── Zoom: 30s before vs 30s during seizure ─────────────────────────────────
onset_sec  = seizure_intervals[0][0]
offset_sec = seizure_intervals[0][1]
pad        = 30

pre_start  = max(0, int((onset_sec - pad) * FS))
pre_end    = int(onset_sec * FS)
sz_start   = int(onset_sec * FS)
sz_end     = min(n_samples, int((onset_sec + pad) * FS))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
ch_idx    = 0   # plot channel 0

# Pre-seizure
t_pre = np.arange(pre_end - pre_start) / FS
axes[0].plot(t_pre, data[ch_idx, pre_start:pre_end], color='steelblue', lw=0.7)
axes[0].set_title(f'Pre-seizure  ({pad}s before onset) — {raw.ch_names[ch_idx]}')
axes[0].set_ylabel('Amplitude (µV)')

# During seizure
t_sz = np.arange(sz_end - sz_start) / FS
axes[1].plot(t_sz, data[ch_idx, sz_start:sz_end], color='tomato', lw=0.7)
axes[1].set_title(f'During seizure  ({pad}s from onset) — {raw.ch_names[ch_idx]}')
axes[1].set_ylabel('Amplitude (µV)'); axes[1].set_xlabel('Time (s)')

plt.tight_layout(); plt.show()

In [ ]:
# ── Amplitude distribution: seizure vs non-seizure ─────────────────────────
sz_mask   = label_array == 1
non_mask  = label_array == 0

sz_amp    = data[:, sz_mask].flatten()
non_amp   = data[:, non_mask].flatten()

# clip for readability
clip = 500
sz_amp  = sz_amp[(sz_amp > -clip) & (sz_amp < clip)]
non_amp = non_amp[(non_amp > -clip) & (non_amp < clip)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(non_amp, bins=200, alpha=0.6, color='steelblue', label='Non-seizure', density=True)
ax.hist(sz_amp,  bins=200, alpha=0.6, color='tomato',    label='Seizure',     density=True)
ax.set_xlabel('Amplitude (µV)'); ax.set_ylabel('Density')
ax.set_title('Amplitude Distribution — Seizure vs Non-Seizure')
ax.legend(); plt.tight_layout(); plt.show()

print(f'Non-seizure: mean={non_amp.mean():.2f} µV,  std={non_amp.std():.2f} µV')
print(f'Seizure    : mean={sz_amp.mean():.2f} µV,  std={sz_amp.std():.2f} µV')

In [ ]:
# ── Seizure duration histogram across all patients ─────────────────────────
all_durations = []
for pdir in patient_dirs:
    ppath = os.path.join(DATA_DIR, pdir)
    summaries = [f for f in os.listdir(ppath) if 'summary' in f.lower() and f.endswith('.txt')]
    if not summaries: continue
    ann_ = parse_summary(os.path.join(ppath, summaries[0]))
    for seiz_list in ann_.values():
        for o, e in seiz_list:
            all_durations.append(e - o)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(all_durations, bins=30, color='mediumseagreen', edgecolor='white')
ax.axvline(np.mean(all_durations), color='red', linestyle='--', label=f'Mean: {np.mean(all_durations):.1f}s')
ax.set_xlabel('Seizure Duration (s)'); ax.set_ylabel('Count')
ax.set_title('Seizure Duration Distribution (all patients)')
ax.legend(); plt.tight_layout(); plt.show()

print(f'Seizures : n={len(all_durations)}')
print(f'Min      : {min(all_durations):.1f}s')
print(f'Max      : {max(all_durations):.1f}s')
print(f'Mean     : {np.mean(all_durations):.1f}s')
print(f'Median   : {np.median(all_durations):.1f}s')

---
## 4. EDA — Frequency Analysis

In [ ]:
# ── Power Spectral Density: seizure vs non-seizure ─────────────────────────
ch_idx = 0

# Take 30s segments
seg_len  = 30 * FS
sz_seg   = data[ch_idx, sz_start : sz_start + seg_len]
non_seg  = data[ch_idx, pre_start : pre_start + seg_len]

f_sz,  psd_sz  = welch(sz_seg,  fs=FS, nperseg=FS*2)
f_non, psd_non = welch(non_seg, fs=FS, nperseg=FS*2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(f_non, psd_non, color='steelblue', label='Non-seizure', lw=1.5)
ax.semilogy(f_sz,  psd_sz,  color='tomato',    label='Seizure',     lw=1.5)
ax.set_xlim(0, 50); ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Power (µV²/Hz)')
ax.set_title(f'Power Spectral Density — {raw.ch_names[ch_idx]}')
# Mark EEG bands
for band, (lo, hi, c) in {'δ(0-4)':(0,4,'purple'), 'θ(4-8)':(4,8,'blue'),
                           'α(8-13)':(8,13,'green'), 'β(13-30)':(13,30,'orange')}.items():
    ax.axvspan(lo, hi, alpha=0.07, color=c, label=band)
ax.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

In [ ]:
# ── Spectrogram centred on seizure onset ───────────────────────────────────
ch_idx  = 0
win_sec = 60   # show ±60s around onset
s_start = max(0, int((onset_sec - win_sec) * FS))
s_end   = min(n_samples, int((onset_sec + win_sec) * FS))
segment = data[ch_idx, s_start:s_end]

f, t_spec, Sxx = spectrogram(segment, fs=FS, nperseg=FS, noverlap=FS//2)
t_spec_abs = t_spec + (s_start / FS)   # absolute time

fig, ax = plt.subplots(figsize=(14, 4))
pcm = ax.pcolormesh(t_spec_abs, f[f <= 50], 10*np.log10(Sxx[f <= 50] + 1e-12),
                    shading='gouraud', cmap='inferno')
for o_, e_ in seizure_intervals:
    ax.axvline(o_, color='cyan', lw=1.5, linestyle='--', label='Seizure onset')
    ax.axvline(e_, color='lime', lw=1.5, linestyle='--', label='Seizure offset')
plt.colorbar(pcm, ax=ax, label='Power (dB)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Frequency (Hz)')
ax.set_title(f'Spectrogram — {raw.ch_names[ch_idx]}  (±{win_sec}s around seizure onset)')
handles, labels_ = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels_[:2], loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── Class imbalance — seizure vs non-seizure windows estimate ──────────────
total_windows  = (n_samples - WINDOW_SAMP) // STEP_SAMP + 1
seizure_samp   = label_array.sum()
sz_windows_est = int(seizure_samp / STEP_SAMP)
non_sz_est     = total_windows - sz_windows_est

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Non-Seizure', 'Seizure'], [non_sz_est, sz_windows_est],
              color=['steelblue', 'tomato'], edgecolor='white', width=0.5)
for bar, val in zip(bars, [non_sz_est, sz_windows_est]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:,}', ha='center', fontsize=10)
ax.set_ylabel('Number of 4s Windows')
ax.set_title(f'Class Imbalance — {TARGET_EDF}\nSeizure: {100*sz_windows_est/total_windows:.1f}%')
plt.tight_layout(); plt.show()

print(f'Total windows   : {total_windows:,}')
print(f'Seizure windows : {sz_windows_est:,}  ({100*sz_windows_est/total_windows:.2f}%)')
print(f'Class weight for seizure: ~{non_sz_est/max(sz_windows_est,1):.0f}x')

---
## 5. Preprocessing Pipeline

In [ ]:
# ── Step 1: Bandpass filter (0.5–40 Hz) ───────────────────────────────────
def bandpass(data, fs, low=0.5, high=40.0, order=4):
    nyq  = fs / 2
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return np.array([filtfilt(b, a, ch) for ch in data])

def notch(data, fs, freq=60.0, Q=30):
    b, a = iirnotch(freq, Q, fs)
    return np.array([filtfilt(b, a, ch) for ch in data])

data_bp   = bandpass(data, FS)
data_filt = notch(data_bp, FS)

# Show before vs after filtering for one channel
ch_idx  = 0
t_plot  = np.arange(10 * FS) / FS   # first 10s

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
axes[0].plot(t_plot, data[ch_idx, :len(t_plot)],      lw=0.6, color='gray')
axes[0].set_title('Raw EEG'); axes[0].set_ylabel('µV')

axes[1].plot(t_plot, data_bp[ch_idx, :len(t_plot)],   lw=0.6, color='steelblue')
axes[1].set_title('After Bandpass (0.5–40 Hz)'); axes[1].set_ylabel('µV')

axes[2].plot(t_plot, data_filt[ch_idx, :len(t_plot)], lw=0.6, color='mediumseagreen')
axes[2].set_title('After Notch (60 Hz removed)'); axes[2].set_ylabel('µV')
axes[2].set_xlabel('Time (s)')

plt.suptitle(f'Filtering Pipeline — {raw.ch_names[ch_idx]}', fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Step 2: Window segmentation + labeling ────────────────────────────────
def make_windows(data, labels, win=WINDOW_SAMP, step=STEP_SAMP, thresh=0.5):
    n_ch, n_samp = data.shape
    n_win = (n_samp - win) // step + 1
    X = np.empty((n_win, n_ch, win), dtype=np.float32)
    y = np.empty(n_win, dtype=np.int8)
    for i in range(n_win):
        s, e = i*step, i*step+win
        X[i]  = data[:, s:e]
        y[i]  = 1 if labels[s:e].mean() >= thresh else 0
    return X, y

windows, win_labels = make_windows(data_filt, label_array)

print(f'Windows shape  : {windows.shape}   (n_windows × channels × samples)')
print(f'Labels shape   : {win_labels.shape}')
print(f'Seizure windows: {win_labels.sum()} / {len(win_labels)}  ({100*win_labels.mean():.2f}%)')

In [ ]:
# ── Step 3: Z-score normalization per channel per window ──────────────────
def zscore(windows, eps=1e-8):
    mu  = windows.mean(axis=-1, keepdims=True)
    std = windows.std(axis=-1, keepdims=True)
    return (windows - mu) / (std + eps)

windows_norm = zscore(windows)

print(f'Before normalization — mean: {windows.mean():.2f},  std: {windows.std():.2f}')
print(f'After  normalization — mean: {windows_norm.mean():.4f},  std: {windows_norm.std():.4f}')

In [ ]:
# ── Visualise a seizure window vs a normal window ──────────────────────────
sz_idx  = np.where(win_labels == 1)[0][0]
non_idx = np.where(win_labels == 0)[0][0]
t_win   = np.arange(WINDOW_SAMP) / FS
offset  = 4   # normalised units offset between channels

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for c in range(min(6, n_channels)):
    axes[0].plot(t_win, windows_norm[non_idx, c] + c*offset, lw=0.7, alpha=0.8)
    axes[1].plot(t_win, windows_norm[sz_idx,  c] + c*offset, lw=0.7, alpha=0.8)

axes[0].set_title('Non-Seizure Window (normalised)')
axes[1].set_title('Seizure Window (normalised)')
for ax in axes:
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude (z-score)')
plt.suptitle('Preprocessed Windows — first 6 channels', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── Sample table: first 5 windows ─────────────────────────────────────────
df_windows = pd.DataFrame({
    'Window':         range(5),
    'Label':          ['SEIZURE' if l else 'Normal' for l in win_labels[:5]],
    'Mean (all ch)':  [round(float(windows_norm[i].mean()), 4) for i in range(5)],
    'Std (all ch)':   [round(float(windows_norm[i].std()),  4) for i in range(5)],
    'Max amplitude':  [round(float(windows_norm[i].max()),  4) for i in range(5)],
})
print('First 5 windows after preprocessing:')
df_windows

---
## 6. Final Dataset Summary

---
## 6. Save Preprocessed Data to Disk

In [ ]:

import time
from tqdm.notebook import tqdm

# ── Where preprocessed .npz files will be saved ───────────────────────────
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── Patient splits (patient-independent) ─────────────────────────────────
SPLITS = {
    'train': list(range(1, 19)),   # chb01 – chb18
    'val':   [19, 20, 21],         # chb19 – chb21
    'test':  [22, 23, 24],         # chb22 – chb24
}

print(f'Saving preprocessed windows to: {os.path.abspath(PROCESSED_DIR)}')
print(f'Train patients : chb01–chb18')
print(f'Val   patients : chb19–chb21')
print(f'Test  patients : chb22–chb24')


In [ ]:

def process_patient(patient_id):
    """
    Loads all available EDF files for one patient,
    runs the full pipeline (filter → window → normalize),
    and returns (windows, labels, patient_id_array).
    """
    pdir        = f'chb{patient_id:02d}'
    ppath       = os.path.join(DATA_DIR, pdir)

    if not os.path.isdir(ppath):
        print(f'  [{pdir}] Directory not found — skipping')
        return None, None, None

    summaries = [f for f in os.listdir(ppath) if 'summary' in f.lower() and f.endswith('.txt')]
    if not summaries:
        print(f'  [{pdir}] No summary file — skipping')
        return None, None, None

    ann_   = parse_summary(os.path.join(ppath, summaries[0]))
    edfs   = sorted([f for f in os.listdir(ppath) if f.endswith('.edf')])

    all_windows, all_labels = [], []

    for edf_file in edfs:
        edf_path = os.path.join(ppath, edf_file)
        try:
            raw_   = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
            d, _   = raw_.get_data(return_times=True)
            d      = d * 1e6   # V → µV

            # resample if needed
            actual_fs = int(raw_.info['sfreq'])
            if actual_fs != FS:
                raw_.resample(FS); d, _ = raw_.get_data(return_times=True); d = d * 1e6

            # build label array
            lbl = np.zeros(d.shape[1], dtype=np.int8)
            for o, e in ann_.get(edf_file, []):
                lbl[int(o*FS):int(e*FS)] = 1

            # filter → window → normalize
            d_filt = notch(bandpass(d, FS), FS)
            wins, lbls = make_windows(d_filt, lbl)
            if wins.shape[0] == 0:
                continue
            wins = zscore(wins)

            all_windows.append(wins)
            all_labels.append(lbls)

        except Exception as ex:
            print(f'  [{pdir}] {edf_file} failed: {ex}')
            continue

    if not all_windows:
        return None, None, None

    W = np.concatenate(all_windows, axis=0)
    L = np.concatenate(all_labels,  axis=0)
    P = np.full(len(L), patient_id, dtype=np.int32)
    return W, L, P


# ── Process all splits and save .npz ─────────────────────────────────────
split_stats = {}

for split_name, patient_ids in SPLITS.items():
    save_path = os.path.join(PROCESSED_DIR, f'{split_name}.npz')

    # Skip if already saved
    if os.path.exists(save_path):
        print(f'\n[{split_name}] Already exists at {save_path} — loading stats...')
        d = np.load(save_path)
        W, L = d['windows'], d['labels']
        split_stats[split_name] = {
            'windows': len(L), 'seizure': int(L.sum()),
            'fraction': round(float(L.mean()), 4), 'size_mb': round(os.path.getsize(save_path)/1e6, 1)
        }
        continue

    print(f'\n[{split_name}]  patients: {[f"chb{p:02d}" for p in patient_ids]}')
    t0 = time.time()

    split_windows, split_labels, split_pids = [], [], []

    for pid in tqdm(patient_ids, desc=f'  {split_name}'):
        W, L, P = process_patient(pid)
        if W is not None:
            split_windows.append(W)
            split_labels.append(L)
            split_pids.append(P)

    if not split_windows:
        print(f'  No data found for {split_name} split — check DATA_DIR')
        continue

    W_all = np.concatenate(split_windows, axis=0)
    L_all = np.concatenate(split_labels,  axis=0)
    P_all = np.concatenate(split_pids,    axis=0)

    # Save compressed
    np.savez_compressed(save_path, windows=W_all, labels=L_all, patient_ids=P_all)

    size_mb = os.path.getsize(save_path) / 1e6
    elapsed = time.time() - t0

    split_stats[split_name] = {
        'windows': len(L_all), 'seizure': int(L_all.sum()),
        'fraction': round(float(L_all.mean()), 4), 'size_mb': round(size_mb, 1)
    }
    print(f'  Saved → {save_path}')
    print(f'  Windows: {len(L_all):,}  |  Seizure: {int(L_all.sum()):,}  ({100*L_all.mean():.2f}%)  |  {size_mb:.0f} MB  |  {elapsed:.0f}s')

print('\nDone.')


In [ ]:

# ── Final summary table ────────────────────────────────────────────────────
print(f'\n{"="*58}')
print(f'{"SAVED FILES":^58}')
print(f'{"="*58}')
print(f'  Location: {os.path.abspath(PROCESSED_DIR)}')
print()
print(f'  {"Split":<8} {"File":<18} {"Windows":>10} {"Seizure":>10} {"Fraction":>10} {"Size":>8}')
print(f'  {"-"*56}')
for split, s in split_stats.items():
    fname = f'{split}.npz'
    print(f'  {split:<8} {fname:<18} {s["windows"]:>10,} {s["seizure"]:>10,} '
          f'{s["fraction"]:>10.2%} {s["size_mb"]:>6.0f} MB')

print(f'\n  Arrays inside each .npz:')
print(f'    windows     → shape (N, channels, 1024)  float32')
print(f'    labels      → shape (N,)                 int8  (0=normal, 1=seizure)')
print(f'    patient_ids → shape (N,)                 int32')
print(f'\n  Load with:')
print(f'    d = np.load("data/processed/train.npz")')
print(f'    X, y = d["windows"], d["labels"]')


In [ ]:
# ── Summary table for all patients on disk ────────────────────────────────
summary_rows = []
for pdir in patient_dirs:
    ppath = os.path.join(DATA_DIR, pdir)
    summaries = [f for f in os.listdir(ppath) if 'summary' in f.lower() and f.endswith('.txt')]
    edfs      = [f for f in os.listdir(ppath) if f.endswith('.edf')]
    if not summaries or not edfs: continue

    ann_  = parse_summary(os.path.join(ppath, summaries[0]))
    sz_edfs_on_disk = [f for f, s in ann_.items() if s and f in edfs]
    all_sz = [s for seiz in ann_.values() for s in seiz]
    total_sz_sec   = sum(e - o for o, e in all_sz)
    # Estimate windows (rough, based on file count)
    est_windows    = len(edfs) * 3600 // STEP_SAMP
    est_sz_windows = int(total_sz_sec * FS / STEP_SAMP)

    summary_rows.append({
        'Patient':          pdir,
        'EDFs on disk':     len(edfs),
        'Seizure events':   len(all_sz),
        'Seizure EDFs':     len(sz_edfs_on_disk),
        'Total seizure (s)':round(total_sz_sec, 1),
        'Est. sz windows':  est_sz_windows,
        'Est. class weight':round(est_windows / max(est_sz_windows, 1), 1),
    })

df_final = pd.DataFrame(summary_rows)
print('Dataset ready for model training:')
df_final

In [ ]:
print('='*50)
print('PREPROCESSING PIPELINE VERIFIED')
print('='*50)
print(f'  Input  : raw EDF  shape {data.shape}  (µV)')
print(f'  Filtered: bandpass 0.5–40Hz + notch 60Hz')
print(f'  Windows: {windows_norm.shape}  (n × channels × {WINDOW_SAMP} samples)')
print(f'  Labels : {win_labels.shape}  — {win_labels.sum()} seizure / {(win_labels==0).sum()} normal')
print(f'  Normalised: mean={windows_norm.mean():.4f}, std={windows_norm.std():.4f}')
print(f'  Ready for CNN+GRU baseline training.')